# deepVogue — Colab Pipeline (GitHub-clone)

**Latent cinema + tarot StyleGAN.** All data lives on Google Drive; the repo is cloned fresh from GitHub each session. Every step is a `make` target so the same commands run locally.

Pipeline: `prepare → train → project anchors → walk → (optional) factors / blend → eval`.

## How to run
1. Runtime → Change runtime type → **GPU (T4 or A100)** → Save
2. Edit cell **00** if your repo URL or branch differs from the defaults, then run it — clones the repo and installs deps via `make colab-install`
3. Edit cell **01** (`DATASET`, `MODE`, `RES`, `FPS`) and run it
4. Run the rest in order, or skip to `pipeline-stills` / `pipeline-frames` for one-shot execution

| Stage | Make target | Output |
|---|---|---|
| 00 Setup       | `colab-clone`, `colab-install` | repo at `/content/deepVogue`, deps installed |
| 01 Configure   | (env)                          | `DV_*` env vars exported |
| 02 Prepare     | `prepare-stills` / `prepare-frames` | `dataset.zip` (+ `frames_index.json` for movies) |
| 03 Train       | `train`                        | `network-snapshot-*.pkl` mirrored to Drive every minute |
| 04 Project     | `project-frames`               | `anchors/<id>/projected_w.npz` per frame |
| 05 Walk        | `walk`                         | `walks/walk.mp4` |
| 06 Factors     | `factors-discover` + `walk-factor` | factor-edited walk |
| 07 Eval        | `eval`                         | FID curve |
| —              | `pipeline-stills`              | full stills run end-to-end |
| —              | `pipeline-frames`              | full latent-cinema run end-to-end |


---
## 00 — Setup (Drive mount + GitHub clone + deps)

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── 0a. config ────────────────────────────────────────────────────────────
GITHUB_URL    = 'https://github.com/JuanGarassino/deepVogue.git'   # public; for private, embed token: https://<TOKEN>@github.com/...
GITHUB_BRANCH = 'master'
REPO_DIR      = '/content/deepVogue'
DP_GITHUB_URL = 'https://github.com/JuanGarassino/dataPalette.git' # consumed by `make colab-install`

# ── 0b. GPU + Drive ───────────────────────────────────────────────────────
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', (r.stdout.strip() or '(none — CPU)'))
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE = Path(os.environ.get('DV_LOCAL_DRIVE', '/tmp/MyDrive'))
    DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE)

# ── 0c. clone (or fast-forward) repo ──────────────────────────────────────
if Path(REPO_DIR + '/.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{GITHUB_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'])

# ── 0d. install via Make ──────────────────────────────────────────────────
os.environ['DV_DATAPALETTE_URL'] = f'git+{DP_GITHUB_URL}'
!make colab-install
print('✓ setup complete')


---
## 01 — Configure run (DV_* env vars)

Pick `DATASET` and `MODE`. Every Make target in this notebook reads these via `deepVogue/_paths.py`.

In [ ]:
import os, datetime

DATASET = 'tarot'      # 'tarot' | 'frames' | <your name>
MODE    = 'stills'     # 'stills' for tarot/fashion, 'frames' for movies
RES     = 512
FPS     = 1            # only used for MODE='frames'
KIMG    = 5000         # training duration (~ images shown / 1000)

STAMP   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ROOT    = str(DRIVE / 'deepVogue')

os.environ['DV_DATA_DIR']    = f'{ROOT}/data/{DATASET}'
os.environ['DV_DATASET_DIR'] = f'{ROOT}/datasets/{DATASET}_{RES}'
os.environ['DV_RUN_DIR']     = f'/content/runs/{DATASET}_{STAMP}'
os.environ['DV_DRIVE_SYNC']  = f'{ROOT}/runs/{DATASET}_{STAMP}'
os.environ['DV_ANCHORS_DIR'] = f'{ROOT}/anchors/{DATASET}_{STAMP}'
os.environ['DV_WALKS_DIR']   = f'{ROOT}/walks/{DATASET}_{STAMP}'
# os.environ['DV_NETWORK_PKL'] = f'{ROOT}/checkpoints/<some>.pkl'  # set to resume / project / walk against an existing model

os.environ['DV_RES']  = str(RES)
os.environ['DV_FPS']  = str(FPS)
os.environ['DV_KIMG'] = str(KIMG)

!make env


---
## 02 — Prepare dataset

In [ ]:
from pathlib import Path
if (Path(os.environ['DV_DATASET_DIR']) / 'dataset.zip').exists():
    print('✓ dataset.zip already exists; skipping.')
else:
    !make prepare-{MODE}


---
## 03 — Train (with background Drive sync)

In [ ]:
from pathlib import Path
from deepVogue._paths import resolve
from deepVogue._drive_sync import DriveSync
p = resolve()
Path(p.run_dir).mkdir(parents=True, exist_ok=True)
sync = DriveSync(p.run_dir, p.drive_sync, interval=60).start()
try:
    !make train
finally:
    sync.stop(final_flush=True)

# adopt the latest checkpoint as DV_NETWORK_PKL for downstream cells
import subprocess
r = subprocess.run(['make', '-s', 'latest-pkl'], capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    os.environ['DV_NETWORK_PKL'] = r.stdout.strip()
    print('DV_NETWORK_PKL =', os.environ['DV_NETWORK_PKL'])


---
## 04 — Project anchors (latent cinema only — skip for stills)

In [ ]:
if MODE == 'frames':
    os.environ['DV_PROJ_STRIDE'] = '4'   # project every 4th anchor; 1 = every frame (slow)
    os.environ['DV_PROJ_STEPS']  = '500'
    !make project-frames
else:
    print('skip — stills mode')


---
## 05 — Render the walk

In [ ]:
from pathlib import Path
target = 'walk' if MODE == 'frames' else 'walk-stills'
!make {target}
out_mp4 = Path(os.environ['DV_WALKS_DIR']) / 'walk.mp4'
from IPython.display import Video, display
if out_mp4.exists():
    display(Video(str(out_mp4), embed=True, width=480))


---
## 06 — Factor edits (optional, Refik-style)

In [ ]:
os.environ['DV_FACTOR_IDX'] = '7'
os.environ['DV_FACTOR_AMP'] = '0.4'
!make factors-discover walk-factor


---
## 07 — Eval (FID curve)

In [ ]:
!make eval


---
## One-shot pipelines

Once cell 01 has set the env, you can collapse 02–07 into a single command:

```bash
!make pipeline-stills    # tarot / fashion
!make pipeline-frames    # latent cinema (prepare → train → project → walk → eval)
```

Both pipelines respect `DV_NETWORK_PKL` (resume), `DV_KIMG`, `DV_PROJ_STRIDE`, `DV_INTERP`, etc.